# **LangChain + MCP**

Este notebook demonstra a integração do LangChain com MCP (Model Context Protocol).

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai.chat_models.base import ChatOpenAI

In [11]:
model = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("BASE_URL")
)

## PASSO 1: Criar e iniciar um servidor MCP local

Execute em um terminal:

```bash
python server.py
```

O código do servidor está na célula abaixo.

```python
from mcp.server.fastmcp import FastMCP
import uvicorn

mcp = FastMCP("Demo Server")

@mcp.tool()
def get_weather(location: str) -> str:
    """Retorna o clima para uma localização."""
    return f"O clima em {location} é ensolarado com 25°C."

@mcp.tool()
def calculate(a: float, b: float, operation: str) -> float:
    """Calculadora simples: +, -, *, /"""
    ops = {"+": a + b, "-": a - b, "*": a * b, "/": a / b if b != 0 else "Erro"}
    return ops.get(operation, "Operação inválida")

if __name__ == "__main__":
    uvicorn.run(mcp.streamable_http_app(), host="0.0.0.0", port=8000)
```

## PASSO 2: Conectar ao servidor MCP local

In [ ]:
MCP_SERVER_URL = "http://localhost:8000/mcp"

client = MultiServerMCPClient(
    {
        "server": {
            "transport": "http",
            "url": MCP_SERVER_URL,
        }
    }
)

tools = await client.get_tools()

print(tools)

[StructuredTool(name='get_weather', description='Retorna o clima para uma localização.', args_schema={'properties': {'location': {'title': 'Location', 'type': 'string'}}, 'required': ['location'], 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7d86b4fdae50>), StructuredTool(name='calculate', description='Calculadora simples: +, -, *, /', args_schema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}, 'operation': {'title': 'Operation', 'type': 'string'}}, 'required': ['a', 'b', 'operation'], 'title': 'calculateArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7d86afb12560>)]


## PASSO 4: Agent ReAct com LangGraph

In [60]:
client = MultiServerMCPClient(
    {
        "server": {
            "transport": "http",
            "url": MCP_SERVER_URL,
        }
    }
)

tools = await client.get_tools()
    
print(tools)

[StructuredTool(name='get_weather', description='Retorna o clima para uma localização.', args_schema={'properties': {'location': {'title': 'Location', 'type': 'string'}}, 'required': ['location'], 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x71d82046be20>), StructuredTool(name='calculate', description='Calculadora simples: +, -, *, /', args_schema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}, 'operation': {'title': 'Operation', 'type': 'string'}}, 'required': ['a', 'b', 'operation'], 'title': 'calculateArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x71d8204ecbf0>)]


In [ ]:
from langchain.agents import create_agent

agent = create_agent(model, tools)

response = await agent.ainvoke(
    {"messages": [("human", "Some 3 + 5 e multiplique o resultado por 12.")]}
)

for msg in response["messages"]:
    print(f"{msg.type}: {msg.content}")

human: Some 3 + 5 e multiplique o resultado por 12.
ai: 
tool: [{'type': 'text', 'text': '8.0', 'id': 'lc_2700ce54-07a9-403f-ad79-dbbf2da0504f'}]
ai: 
tool: [{'type': 'text', 'text': '96.0', 'id': 'lc_0b2c4f4b-f6f6-41c1-bb49-27e3b410a1ea'}]
ai: 3 + 5 = 8, e 8 × 12 = 96. Portanto o resultado é 96.
